# PPO with Stable-Baselines3 in 15 Minutes
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/10_Reinforcement_Learning/ppo_stable_baselines3_intro.ipynb)

PPO (Proximal Policy Optimization) learns a POLICY directly and clips policy updates so training never takes destructive steps - simple, stable, and the default choice in industry RL (and famously in RLHF).

Stable-Baselines3 gives battle-tested implementations in a few lines.

In [ ]:
!pip install -q stable-baselines3 gymnasium

## 1. Baseline: untrained agent

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

env = Monitor(gym.make("CartPole-v1"))

def random_score(episodes=20):
    scores = []
    for _ in range(episodes):
        s, _ = env.reset(); total, done = 0, False
        while not done:
            s, r, term, trunc, _ = env.step(env.action_space.sample())
            total += r; done = term or trunc
        scores.append(total)
    return np.mean(scores)

print(f"random policy: {random_score():.1f} avg reward")

## 2. Train PPO

In [ ]:
model = PPO("MlpPolicy", env, verbose=0,
            learning_rate=3e-4, n_steps=1024, batch_size=64,
            gamma=0.99, seed=42)
model.learn(total_timesteps=100_000, progress_bar=False)
print("training done")

## 3. Evaluate the trained policy properly

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20)
print(f"PPO policy : {mean_r:.1f} +/- {std_r:.1f}   (solved threshold = 475)")

## 4. Watch it + save/load

In [ ]:
save_note = """
model.save("ppo_cartpole")                    # zip artifact
model = PPO.load("ppo_cartpole")               # reload anywhere

# render one episode locally:
env_r = gym.make("CartPole-v1", render_mode="human")
obs, _ = env_r.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, r, term, trunc, _ = env_r.step(int(action))
    done = term or trunc
"""
print(save_note)

## PPO in one paragraph
Collect rollouts with the current policy -> estimate advantages -> update policy to increase the probability of good actions, but CLIP the probability ratio to [1-e, 1+e] so no single update moves too far -> repeat.

| Knob | Default | Meaning |
|---|---|---|
| `n_steps` | 1024 | rollout length per update |
| `learning_rate` | 3e-4 | policy net step size |
| clip range | 0.2 | max policy change per update |

Same API swaps environments: `"LunarLander-v3"`, Atari CNN policies, or your custom `gym.Env`.